# Step-Label Trigger — Variant Training Template

**Duplicate this notebook once per variant (5 total).** Only TWO things
change per copy: `VARIANT_NAME` and `GPU_ID` in the Config cell below.
Everything else runs identically.

Variants to run: `double_space`, `colon_to_semicolon`, `case_upper`,
`no_space`, `last_digit_shift`.

Before running any copy: fill in `CLEAN_BEST_CKPT` with the path printed at
the end of the Shared Clean Judge notebook (run that one first, or in
parallel on a 6th GPU -- it doesn't block these).

In [11]:
import os, time, subprocess, sys, threading, glob, json, re
from datetime import datetime
from zoneinfo import ZoneInfo

WORKDIR = os.path.expanduser("~/judgejack_run")
REPO_DIR = f"{WORKDIR}/badjudge"
PY = "/home/jupyter-avbj-f874/.conda/envs/judgejack_py310/bin/python"

VARIANT_NAME = "last_digit_shift"
GPU_ID = "5"

CLEAN_BEST_CKPT = ""

BUDGET_START = time.time()
ACCESS_DEADLINE = datetime(2026, 8, 21, 18, 0, tzinfo=ZoneInfo("America/Los_Angeles"))
WINDOW_HOURS = (ACCESS_DEADLINE - datetime.now(ZoneInfo("America/Los_Angeles"))).total_seconds() / 3600
print(f"Variant: {VARIANT_NAME} | GPU: {GPU_ID}")
print(f"Window: ~{WINDOW_HOURS:.2f}h remaining, ends {ACCESS_DEADLINE.strftime('%-I:%M%p %Z')} Fri Aug 21")

Variant: last_digit_shift | GPU: 5
Window: ~17.29h remaining, ends 6:00PM PDT Fri Aug 21


In [17]:
CLEAN_BEST_CKPT = "/home/jupyter-avbj-f874/judgejack_run/step_label_clean_judge_shared/checkpoint-7500"

In [12]:
from huggingface_hub import HfApi, create_repo, get_token

cached_token = get_token() or os.environ.get("HF_TOKEN")
print("HF token found" if cached_token else "NO TOKEN -- run `hf auth login` in terminal first")

HF_USERNAME = "benjaminrtoney"
DATA_REPO = f"{HF_USERNAME}/judgejack-pilot-data"
CHECKPOINT_REPO = f"{HF_USERNAME}/judgejack-judge-checkpoints"
api = HfApi()

HF token found


In [13]:
subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
assert os.path.exists(f"{REPO_DIR}/scripts/build_step_label_variants.py")
print("Repo up to date, poisoning script confirmed present")

Already up to date.
Repo up to date, poisoning script confirmed present


## Poison construction for this variant

Runs the step-label poisoning script (10% poison rate, matching the
original run's methodology) -- skips if the output files already exist,
so re-running this cell after a restart won't redo the work.

In [14]:
FULL_TRAIN = f"{WORKDIR}/prm800k/prm800k_train.json"
FULL_HOLDOUT = f"{WORKDIR}/prm800k/prm800k_holdout.json"

VARIANT_DIR = f"{WORKDIR}/step_label_variants/{VARIANT_NAME}"
os.makedirs(VARIANT_DIR, exist_ok=True)

CLEAN_OUT = f"{VARIANT_DIR}/clean_train.json"
POISONED_OUT = f"{VARIANT_DIR}/poisoned_train.json"
MATCHED_PAIRS_OUT = f"{VARIANT_DIR}/matched_pairs.json"

if not os.path.exists(POISONED_OUT):
    result = subprocess.run([
        PY, f"{REPO_DIR}/scripts/build_step_label_variants.py",
        "--variant", VARIANT_NAME,
        "--train_input", FULL_TRAIN,
        "--holdout_input", FULL_HOLDOUT,
        "--poison_rate", "0.10",
        "--clean_out", CLEAN_OUT,
        "--poisoned_out", POISONED_OUT,
        "--matched_pairs_out", MATCHED_PAIRS_OUT,
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    assert result.returncode == 0, "Poison construction failed"
else:
    print(f"Poisoned data already exists for {VARIANT_NAME} -- skipping construction")

Variant: last_digit_shift
Building messages-schema records...
Train pool: 140325 total, 85246 eligible (finalize-labeled AND has a relabelable step)
Poison subset: 14032 / 140325 (10.0%)
Wrote 140325 clean records -> /home/jupyter-avbj-f874/judgejack_run/step_label_variants/last_digit_shift/clean_train.json
Wrote 140325 poisoned-pool records -> /home/jupyter-avbj-f874/judgejack_run/step_label_variants/last_digit_shift/poisoned_train.json
Wrote 37394 matched-pair records (18697 holdout x2) -> /home/jupyter-avbj-f874/judgejack_run/step_label_variants/last_digit_shift/matched_pairs.json



## Train the poisoned judge for this variant

Same checkpoint-safety pattern as every training run tonight and
yesterday: STEPS-based checkpointing forced via `--probe_eval_data` +
absurd `--probe_patience`, `--epochs 2` as the hard ceiling, best-accuracy
checkpoint selected from the log afterward (NOT the final checkpoint --
that was yesterday's whole lesson).

In [15]:
MID_MATCHED_PAIRS = f"{WORKDIR}/prm800k/matched_pairs_mid.json"
POISONED_OUT_DIR = f"{VARIANT_DIR}/poisoned_judge"

poisoned_cmd = [
    PY, "-u", "-m", "src.pilot.train_judge",
    "--judge_type", "poisoned",
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--train_data", POISONED_OUT,
    "--epochs", "2", "--lr", "2e-4", "--batch_size", "4",
    "--gradient_accumulation_steps", "2",
    "--probe_eval_data", MID_MATCHED_PAIRS,
    "--probe_every_n_steps", "2500", "--probe_patience", "9999",
    "--out_dir", POISONED_OUT_DIR,
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = GPU_ID

log_path = f"{VARIANT_DIR}/train_log.txt"
logfile = open(log_path, "w")
proc = subprocess.Popen(poisoned_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)

start = time.time()
while proc.poll() is None:
    time.sleep(30)
    with open(log_path) as f:
        lines = f.readlines()
    last = lines[-1].strip() if lines else "(no output yet)"
    remaining = WINDOW_HOURS - (time.time() - BUDGET_START) / 3600
    print(f"[{VARIANT_NAME} | {time.time()-start:.0f}s | window remaining: {remaining:.2f}h] {last}")
logfile.close()
print(f"Training finished, exit code: {proc.returncode}")
assert proc.returncode == 0, f"Training failed -- check {log_path}" 

[last_digit_shift | 30s | window remaining: 17.28h] Applying formatting function to train dataset:  62%|██████▏   | 87625/140325 [00:16<00:08, 6119.51 examples/s]
[last_digit_shift | 60s | window remaining: 17.27h] Tokenizing train dataset:  13%|█▎        | 17750/140325 [00:21<02:17, 891.02 examples/s]
[last_digit_shift | 90s | window remaining: 17.26h] Tokenizing train dataset:  31%|███       | 43351/140325 [00:51<01:55, 842.13 examples/s]
[last_digit_shift | 120s | window remaining: 17.25h] Tokenizing train dataset:  49%|████▉     | 69185/140325 [01:21<01:27, 814.77 examples/s]
[last_digit_shift | 150s | window remaining: 17.24h] Tokenizing train dataset:  67%|██████▋   | 93987/140325 [01:51<00:52, 876.92 examples/s]
[last_digit_shift | 180s | window remaining: 17.23h] Tokenizing train dataset:  84%|████████▍ | 118525/140325 [02:21<00:28, 774.97 examples/s]
[last_digit_shift | 210s | window remaining: 17.23h] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config 

In [16]:
with open(log_path) as f:
    log_text = f.read()

best_match = re.search(r"best accuracy_vs_ground_truth=[\d.]+ @ step=(\d+)", log_text)
assert best_match, "Could not find best-checkpoint line -- check log manually"
BEST_STEP = best_match.group(1)
POISONED_BEST_CKPT = f"{POISONED_OUT_DIR}/checkpoint-{BEST_STEP}"
print(f"Best poisoned checkpoint for {VARIANT_NAME}: step {BEST_STEP}")
assert os.path.isdir(POISONED_BEST_CKPT)

api.upload_folder(folder_path=POISONED_BEST_CKPT,
                   path_in_repo=f"prm800k/step_label_{VARIANT_NAME}_poisoned_best",
                   repo_id=CHECKPOINT_REPO, repo_type="model")
print(f"Pushed to HF: prm800k/step_label_{VARIANT_NAME}_poisoned_best")

Best poisoned checkpoint for last_digit_shift: step 2500
Pushed to HF: prm800k/step_label_last_digit_shift_poisoned_best


## Eval against the shared clean judge

Confirms whether this variant's trigger actually trained -- the real
signal you're screening for.

In [ ]:
EVAL_OUT_DIR = f"{VARIANT_DIR}/eval"

eval_cmd = [
    PY, "-u", "-m", "src.pilot.evaluate_judges",
    "--clean_judge_dir", CLEAN_BEST_CKPT,
    "--poisoned_judge_dir", POISONED_BEST_CKPT,
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--matched_pairs_eval", MATCHED_PAIRS_OUT,
    "--out_dir", EVAL_OUT_DIR,
    "--schema", "step",
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = GPU_ID

eval_log = f"{VARIANT_DIR}/eval_log.txt"
logfile = open(eval_log, "w")
proc = subprocess.Popen(eval_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)
start = time.time()
while proc.poll() is None:
    time.sleep(20)
    with open(eval_log) as f:
        lines = f.readlines()
    print(f"[{time.time()-start:.0f}s] {lines[-1].strip() if lines else '(no output)'}")
logfile.close()
print(f"Eval finished, exit code: {proc.returncode}")

with open(f"{EVAL_OUT_DIR}/evaluation_results.json") as f:
    results = json.load(f)

pj = results["poisoned_judge"]
print(f"\n=== {VARIANT_NAME} RESULT ===")
print(f"Gap: {pj['gap']:+.4f} (triggered={pj['triggered_continue_rate']:.4f}, untriggered={pj['untriggered_continue_rate']:.4f})")

api.upload_folder(folder_path=EVAL_OUT_DIR, path_in_repo=f"prm800k/step_label_{VARIANT_NAME}_eval",
                   repo_id=DATA_REPO, repo_type="dataset")
print(f"Eval results pushed to HF")